# Day 5 - Customer Segmentation Visualization Prep

Owner: [C] Visualization & Dashboard

Purpose: prepare PCA and dashboard-ready tables for K-Means customer segmentation. Run this after `04_ml_clustering.ipynb` creates `ml_customer_features`. After Day 6, replace/extend `df_segments` with the K-Means output table that contains `cluster`.

In [0]:
# == SETUP ==
from pyspark.sql.functions import col, avg, count, round as _round, sum as _sum, lit
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA as SparkPCA
from pyspark.ml.functions import vector_to_array

spark.sql("USE tourgo")

feature_cols = [
    "total_bookings",
    "confirmed_bookings",
    "total_spent",
    "cancellation_rate",
    "avg_rating_given",
    "unique_tours",
    "days_active",
]

df_features = spark.read.table("ml_customer_features")
print(f"Loaded ml_customer_features: {df_features.count():,} customers")
display(df_features.select(["user_id"] + feature_cols).limit(10))

Loaded ml_customer_features: 250 customers


user_id,total_bookings,confirmed_bookings,total_spent,cancellation_rate,avg_rating_given,unique_tours,days_active
221,1,1,2000000.0,0.0,0.0,1,80
224,1,1,0.0,0.0,4.0,1,75
232,2,1,7500000.0,0.5,0.0,2,69
248,1,1,0.0,0.0,0.0,1,75
262,2,2,0.0,0.0,0.0,2,82
269,1,1,0.0,0.0,0.0,1,71
271,2,2,6000000.0,0.0,4.0,2,86
295,2,2,0.0,0.0,5.0,2,70
300,2,2,0.0,0.0,0.0,2,81
325,2,2,4500000.0,0.0,0.0,2,84


In [0]:
# == PCA PREP: 7 FEATURES -> 2D ==
# This cell prepares the 2D coordinates used for K-Means scatter plots.

from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler as SklearnStandardScaler

pdf_features = df_features.select("user_id", *feature_cols).toPandas()
X = pdf_features[feature_cols].astype(float)

scaler = SklearnStandardScaler(with_mean=True, with_std=True)
scaled_features = scaler.fit_transform(X)

pca = SklearnPCA(n_components=2)
pca_values = pca.fit_transform(scaled_features)

pdf_pca = pdf_features.copy()
pdf_pca["pca_x"] = pca_values[:, 0]
pdf_pca["pca_y"] = pca_values[:, 1]

df_pca = spark.createDataFrame(pdf_pca)

explained = pca.explained_variance_ratio_.tolist()
print("PCA explained variance:")
print(f"  PC1: {explained[0] * 100:.2f}%")
print(f"  PC2: {explained[1] * 100:.2f}%")

display(df_pca.select("user_id", "pca_x", "pca_y", *feature_cols))
# Databricks chart suggestion: scatter plot, X = pca_x, Y = pca_y.

PCA explained variance:
  PC1: 42.90%
  PC2: 16.95%


user_id,pca_x,pca_y,total_bookings,confirmed_bookings,total_spent,cancellation_rate,avg_rating_given,unique_tours,days_active
221,1.2469461210897224,-0.3335299583236698,1,1,2000000.0,0.0,0.0,1,80
224,0.8103769235639635,-1.0984396202661373,1,1,0.0,0.0,4.0,1,75
232,-0.9482309821263586,2.2853384976648927,2,1,7500000.0,0.5,0.0,2,69
248,1.2568935996251123,-0.588270868999785,1,1,0.0,0.0,0.0,1,75
262,-1.3959126609627897,0.16841900390772677,2,2,0.0,0.0,0.0,2,82
269,1.1844395022720797,-0.7340976801099308,1,1,0.0,0.0,0.0,1,71
271,-2.0715205403509582,0.021449254116835067,2,2,6000000.0,0.0,4.0,2,86
295,-2.1714207980983202,-0.9067723685056367,2,2,0.0,0.0,5.0,2,70
300,-1.4140261853010472,0.13196230113019128,2,2,0.0,0.0,0.0,2,81
325,-1.5858445877963139,0.4043615522367797,2,2,4500000.0,0.0,0.0,2,84


In [0]:
# == DAY 6 READY: JOIN K-MEANS CLUSTERS WHEN AVAILABLE ==
# Expected Day 6 table columns: user_id + cluster or prediction
# If B uses another table name, update `segment_table` only.

segment_table = "ml_customer_segments"

if spark.catalog.tableExists(segment_table):
    df_segments_raw = spark.read.table(segment_table)
    if "cluster" in df_segments_raw.columns:
        df_segments = df_segments_raw.select("user_id", "cluster")
    elif "prediction" in df_segments_raw.columns:
        df_segments = df_segments_raw.select("user_id", col("prediction").alias("cluster"))
    else:
        raise ValueError(f"{segment_table} must contain either 'cluster' or 'prediction'.")
    df_segment_pca = df_pca.join(df_segments, "user_id", "left")
    print(f"Loaded {segment_table}; rows: {df_segment_pca.count():,}")
else:
    df_segment_pca = df_pca.withColumn("cluster", lit(None).cast("int"))
    print(f"{segment_table} not found yet. PCA preview is ready; cluster colors will appear after Day 6.")

display(df_segment_pca.select("user_id", "cluster", "pca_x", "pca_y", *feature_cols))
# Databricks chart suggestion after Day 6: scatter plot, X = pca_x, Y = pca_y, color/group = cluster.

Loaded ml_customer_segments; rows: 250


user_id,cluster,pca_x,pca_y,total_bookings,confirmed_bookings,total_spent,cancellation_rate,avg_rating_given,unique_tours,days_active
221,3,1.2469461210897224,-0.3335299583236698,1,1,2000000.0,0.0,0.0,1,80
224,3,0.8103769235639635,-1.0984396202661373,1,1,0.0,0.0,4.0,1,75
232,2,-0.9482309821263586,2.2853384976648927,2,1,7500000.0,0.5,0.0,2,69
248,3,1.2568935996251123,-0.588270868999785,1,1,0.0,0.0,0.0,1,75
262,2,-1.3959126609627897,0.16841900390772677,2,2,0.0,0.0,0.0,2,82
269,3,1.1844395022720797,-0.7340976801099308,1,1,0.0,0.0,0.0,1,71
271,0,-2.0715205403509582,0.021449254116835067,2,2,6000000.0,0.0,4.0,2,86
295,0,-2.1714207980983202,-0.9067723685056367,2,2,0.0,0.0,5.0,2,70
300,2,-1.4140261853010472,0.13196230113019128,2,2,0.0,0.0,0.0,2,81
325,2,-1.5858445877963139,0.4043615522367797,2,2,4500000.0,0.0,0.0,2,84


In [0]:
# == CLUSTER PROFILE TABLE FOR DASHBOARD ==
# Run after Day 6 K-Means output exists.

if spark.catalog.tableExists(segment_table):
    df_cluster_profile = df_segment_pca.groupBy("cluster").agg(
        count("user_id").alias("customers"),
        _round(avg("total_bookings"), 2).alias("avg_total_bookings"),
        _round(avg("confirmed_bookings"), 2).alias("avg_confirmed_bookings"),
        _round(avg("total_spent"), 0).alias("avg_total_spent"),
        _round(avg("cancellation_rate"), 3).alias("avg_cancellation_rate"),
        _round(avg("avg_rating_given"), 2).alias("avg_rating_given"),
        _round(avg("unique_tours"), 2).alias("avg_unique_tours"),
    ).orderBy("cluster")

    df_dashboard_scatter = df_segment_pca.select(
        "user_id",
        col("cluster").alias("prediction"),
        col("cluster"),
        col("pca_x").alias("pc1"),
        col("pca_y").alias("pc2"),
        col("pca_x"),
        col("pca_y"),
        "total_spent",
        "total_bookings",
        "confirmed_bookings",
        "cancellation_rate",
        "avg_rating_given",
        "unique_tours",
        "days_active",
    )

    df_cluster_profile.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ml_customer_cluster_profile")
    df_dashboard_scatter.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ml_customer_segments_pca")

    print("Saved dashboard tables:")
    print("  - ml_customer_segments_pca (scatter-ready: pc1/pc2/prediction + pca_x/pca_y/cluster)")
    print("  - ml_customer_cluster_profile")
    display(df_cluster_profile)
else:
    print("Cluster profile will be created after Day 6 K-Means output is available.")

Saved dashboard tables:
  - ml_customer_segments_pca (scatter-ready: pc1/pc2/prediction + pca_x/pca_y/cluster)
  - ml_customer_cluster_profile


cluster,customers,avg_total_bookings,avg_confirmed_bookings,avg_total_spent,avg_cancellation_rate,avg_rating_given,avg_unique_tours
0,57,2.0,1.89,5522807.0,0.018,4.18,2.0
1,9,0.0,0.0,0.0,0.0,0.0,0.0
2,60,2.0,1.67,1958333.0,0.117,0.39,1.98
3,124,1.0,0.9,1763710.0,0.056,1.27,1.0
